# Open Text — extract by resource ID

Run **`@istari:extract`** (Open Text / `textract`) on a Model resource you already registered in the Istari Digital web app.

You will:

1. Connect to the Istari Digital Platform
2. Paste the Model **resource UUID** for an uploaded file
3. Submit `@istari:extract` and poll until artifacts appear

Companion to the [Open Text](https://docs.istaridigital.com/integrations/Productivity/open_text) integration. SDK calls match the **Usage → SDK** sample there (`add_job` with `function="@istari:extract"`, `tool_name="textract"`).

> **Prerequisite:** upload a supported file in the Istari Digital web app (**Files**), then copy its resource UUID. Use [`sample.txt`](./sample.txt) in this folder (from the integration example files), or any `.csv`, `.docx`, `.htm`, `.html`, `.json`, `.log`, `.odt`, `.psv`, `.tsv`, or `.txt`.

### Prerequisites

- **`istari-digital-client`** and **`python-dotenv`** (from cookbook root: `uv sync --group dev`).
- Access to **`@istari:extract`** with tool **`textract`** on an agent whose OS matches **Prep**.
- [`samples/.env`](../../.env): `ISTARI_REGISTRY_URL`, `ISTARI_PERSONAL_ACCESS_TOKEN`.

### Install kernel (optional)

From the cookbook repository root:

```bash
uv sync --group dev
uv run python -m ipykernel install --user --name istari-client-cookbook --display-name "Python (istari-client-cookbook)"
```

Select **Python (istari-client-cookbook)** in the kernel picker.

### Running order

Run top to bottom. Edit **Prep** before submitting the job.

> **Pause in the web app:** after upload, copy the Model UUID from **Files**. After §2, open **Jobs** to watch `@istari:extract` on the agent.

## 1 · Connect

Load [`samples/.env`](../../.env), build `Configuration`, and create `Client`.

In [ ]:
import os
from pathlib import Path
from time import sleep

import dotenv
from istari_digital_client import Client, Configuration, JobStatusName

SAMPLES_DIR = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / ".env").exists() and (candidate / "connect").is_dir():
        SAMPLES_DIR = candidate
        break

dotenv.load_dotenv(SAMPLES_DIR / ".env", override=True)
registry_url = os.environ.get("ISTARI_REGISTRY_URL")
token = os.environ.get("ISTARI_PERSONAL_ACCESS_TOKEN")
if not registry_url or not token:
    raise RuntimeError("Set ISTARI_REGISTRY_URL and ISTARI_PERSONAL_ACCESS_TOKEN in samples/.env")

client = Client(Configuration(registry_url=registry_url, registry_auth_token=token))
me = client.get_current_user()
print(f"Registry: {registry_url}")
print(f"Signed in as: {me.display_name} ({me.email})")

## 2 · Prep

Paste the Model UUID from the web app. Match `TOOL_VERSION` and `OPERATING_SYSTEM` to your Istari Digital Agent (see [Open Text](https://docs.istaridigital.com/integrations/Productivity/open_text)).

In [ ]:
# Prerequisite: upload a supported file in the Istari Digital web app (Files),
# then paste its Model resource UUID. Example file: sample.txt in this folder.
RESOURCE_ID = ""  # paste the UUID from the web app

if not RESOURCE_ID.strip():
    raise ValueError(
        "RESOURCE_ID is required — upload the file in the Istari Digital web app "
        "and paste the Model UUID."
    )

TOOL_NAME = "textract"
TOOL_VERSION = "1.6.3"
OPERATING_SYSTEM = "Windows 11"  # or Ubuntu 22.04, RHEL 8, macOS 14, …

print(f"Resource ID: {RESOURCE_ID}")
print(f"Tool: {TOOL_NAME} {TOOL_VERSION} on {OPERATING_SYSTEM}")

## 3 · Submit `@istari:extract`

Same `add_job` shape as the Open Text SDK sample: run extract on the existing model id, poll, then list artifacts (`extracted_text`, `metadata_report`).

In [ ]:
model = client.get_model(RESOURCE_ID)

extraction_job = client.add_job(
    model_id=model.id,
    function="@istari:extract",
    tool_name=TOOL_NAME,
    tool_version=TOOL_VERSION,
    operating_system=OPERATING_SYSTEM,
)
print(f"Extraction started for model ID {model.id}, job ID: {extraction_job.id}")

elapsed = 0
poll_interval = 5
while True:
    extraction_job = client.get_job(extraction_job.id)
    status = extraction_job.status.name
    print(f"{elapsed}s: {status.value}")
    if status in (
        JobStatusName.COMPLETED,
        JobStatusName.FAILED,
        JobStatusName.CANCELED,
    ):
        break
    sleep(poll_interval)
    elapsed += poll_interval

if extraction_job.status.name != JobStatusName.COMPLETED:
    msg = extraction_job.status.message or ""
    raise RuntimeError(f"Job {extraction_job.id} ended with {extraction_job.status.name!s}. {msg}")

model = client.get_model(model.id)
print("\nArtifacts:")
for artifact in model.artifacts or []:
    print(f"  {artifact.name}")

## Learn more

- [Open Text](https://docs.istaridigital.com/integrations/Productivity/open_text) — functions, example files, `@istari:parse_json_fields`
- [Python Client — Quick Start](https://docs.istaridigital.com/developers/SDK/setup)
- [Chaining jobs (official client)](../../chaining_jobs_no_helper.ipynb) — `add_job` plus lineage